# Model Training and Comparison

In this notebook, we train and compare multiple regression models for house price prediction.

Main tasks:

- Load the cleaned dataset
- Split the data into training and testing sets
- Build preprocessing and modeling pipelines
- Train multiple regression models
- Compare model performance using evaluation metrics

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys

import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
sys.path.append(os.path.abspath(".."))

## Load Cleaned Dataset

In [4]:
df = pd.read_csv("../data/processed/cleaned_train.csv")

df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,Reg,Lvl,AllPub,FR2,...,0,0,0,0,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,0,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,IR1,Lvl,AllPub,Corner,...,272,0,0,0,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,IR1,Lvl,AllPub,FR2,...,0,0,0,0,0,12,2008,WD,Normal,250000


## Separate Features and Target

In [5]:
X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

y = np.log1p(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1460, 74)
y shape: (1460,)


## Identify Numerical and Categorical Features

In [6]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Number of numerical features: 37
Number of categorical features: 37


In [7]:
## Train-Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1168, 74)
X_test shape: (292, 74)


## Build Preprocessing Pipelines

In [9]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## Define Regression Models

In [10]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

## Define Evaluation Function

In [11]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Pipeline": pipeline
    }

## Train and Evaluate Models

In [12]:
results = []

for name, model in models.items():
    result = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(result)

## Create Results Table

In [13]:
results_df = pd.DataFrame([
    {
        "Model": r["Model"],
        "MAE": r["MAE"],
        "RMSE": r["RMSE"],
        "R2": r["R2"]
    }
    for r in results
])

results_df = results_df.sort_values(by="RMSE").reset_index(drop=True)
results_df

,Model,MAE,RMSE,R2
0,Ridge,0.091197,0.127026,0.913533
1,Linear Regression,0.089380,0.128658,0.911298
2,Gradient Boosting,0.093907,0.137109,0.899262
3,Random Forest,0.099223,0.145149,0.887101
4,Lasso,0.337134,0.433244,-0.005837


## Select the Best Model

In [14]:
best_result = min(results, key=lambda x: x["RMSE"])
best_model = best_result["Pipeline"]

print("Best Model:", best_result["Model"])
print("Best MAE:", best_result["MAE"])
print("Best RMSE:", best_result["RMSE"])
print("Best R2:", best_result["R2"])

Best Model: Ridge
Best MAE: 0.09119741221870986
Best RMSE: 0.12702637691585072
Best R2: 0.9135329566904389


## Modeling Summary

We trained multiple regression models and compared their performance using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R-squared (R2)

The best-performing model will be used in the next step for final evaluation and saving.